# Wind fleet: generation model, day-ahead forecast error and imbalance exposure

We are being asked to quote a fixed-price PPA on a 10 GW onshore/offshore wind portfolio.
This notebook builds generation from the site wind speed with a power curve, emulates a
day-ahead wind forecast, and sizes the imbalance exposure from forecast error.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

pd.set_option("display.width", 120)
rng = np.random.default_rng(42)

df = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"])
P_RATED = 10_000                      # MW, whole fleet
CUT_IN, V_RATED, CUT_OUT = 3.5, 12.0, 25.0

# met-mast wind is at 10 m; scale to 100 m hub height with a power-law shear profile
SHEAR = 0.14
df["v_hub"] = df["wind_ms"] * (100 / 10) ** SHEAR
df[["wind_ms", "v_hub"]].describe().round(2).T

,count,mean,std,min,25%,50%,75%,max
wind_ms,17520.0,7.30,2.50,0.0,5.56,7.22,8.95,15.99
v_hub,17520.0,10.07,3.45,0.0,7.67,9.97,12.36,22.07


## Power curve

Manufacturer curve, normalised to 1 at rated power, tabulated at a few wind speeds.

In [2]:
curve_pts = {0.0: 0.0, 3.5: 0.0, 25.0: 0.0, 12.0: 1.0, 5.0: 0.07, 7.0: 0.20, 9.0: 0.42, 10.5: 0.67, 11.5: 0.92, 14.0: 1.0}
xp, fp = list(curve_pts.keys()), list(curve_pts.values())

gen_tab = np.interp(df["v_hub"], xp, fp) * P_RATED
print("capacity factor from tabulated curve:", round(gen_tab.mean() / P_RATED, 3))
pd.Series(np.interp(np.arange(0, 26, 2.0), xp, fp), index=np.arange(0, 26, 2.0)).round(2)

capacity factor from tabulated curve: 0.377


0.0     0.0
2.0     0.0
4.0     0.0
6.0     0.0
8.0     0.0
10.0    0.0
12.0    0.0
14.0    0.0
16.0    1.0
18.0    1.0
20.0    1.0
22.0    1.0
24.0    1.0
dtype: float64

The tabulated curve gives an odd capacity factor, presumably from the coarse tabulation
around the knee. Use the analytic cubic between cut-in and rated instead, which is
smooth and standard.

In [3]:
def power_curve(v, v_rated=9.0):
    v = np.asarray(v, dtype=float)
    p = np.clip((v / v_rated) ** 3, 0, 1)
    return np.where(v < CUT_IN, 0.0, p) * P_RATED

df["gen_mw"] = power_curve(df["v_hub"])
cf = df["gen_mw"].mean() / P_RATED
print(f"capacity factor: {cf:.3f}")
df["gen_mw"].describe().round(0)

capacity factor: 0.801


count    17520.0
mean      8008.0
std       3063.0
min          0.0
25%       6202.0
50%      10000.0
75%      10000.0
max      10000.0
Name: gen_mw, dtype: float64

A capacity factor around 70% is consistent with a good offshore-weighted portfolio.

## Day-ahead wind forecast

We do not keep a forecast archive for this site. Emulate the vendor forecast as the
observed wind plus the vendor's quoted error (sd 0.5 m/s).

In [4]:
df["v_fc"] = np.clip(df["v_hub"] + rng.normal(0, 0.5, len(df)), 0, None)
df["gen_fc_mw"] = power_curve(df["v_fc"])
df[["v_hub", "v_fc", "gen_mw", "gen_fc_mw"]].head()

,v_hub,v_fc,gen_mw,gen_fc_mw
0,9.662690,9.815048,10000.0,10000.000000
1,9.124340,8.604348,10000.0,8738.288173
2,9.855944,10.231169,10000.0,10000.000000
3,9.731709,10.201991,10000.0,10000.000000
4,10.159628,9.184111,10000.0,10000.000000


## Align forecast and actuals

The vendor delivers forecasts stamped in UK local time, so the merge is done on the
local wall-clock timestamp.

In [5]:
actual = df[["time", "gen_mw"]].copy()
actual["ts"] = actual["time"].dt.tz_convert("Europe/London").dt.tz_localize(None)

forecast = df[["time", "v_fc", "gen_fc_mw"]].copy()
forecast["ts"] = forecast["time"].dt.tz_localize(None)

m = actual.merge(forecast[["ts", "v_fc", "gen_fc_mw"]], on="ts", how="inner")
print(len(actual), len(forecast), len(m))
m.head(3)

17520 17520 17520


,time,gen_mw,ts,v_fc,gen_fc_mw
0,2022-01-01 00:00:00+00:00,10000.0,2022-01-01 00:00:00,9.815048,10000.000000
1,2022-01-01 01:00:00+00:00,10000.0,2022-01-01 01:00:00,8.604348,8738.288173
2,2022-01-01 02:00:00+00:00,10000.0,2022-01-01 02:00:00,10.231169,10000.000000


## Calibration

Vendor forecasts are usually biased at the site level. Fit a linear correction from
forecast generation to actual generation and use the corrected forecast from here on.

In [6]:
lr = LinearRegression().fit(m[["gen_fc_mw"]], m["gen_mw"])
m["gen_cal_mw"] = lr.predict(m[["gen_fc_mw"]])
print("calibration slope, intercept:", round(lr.coef_[0], 3), round(lr.intercept_, 1))

m["err_mw"] = m["gen_mw"] - m["gen_cal_mw"]
rmse_mw = np.sqrt((m["err_mw"] ** 2).mean())
rmse_pct = rmse_mw / P_RATED
print(f"RMSE: {rmse_mw:,.0f} MW  = {rmse_pct:.1%} of average output")

calibration slope, intercept: 0.92 659.0
RMSE: 1,155 MW  = 11.5% of average output


## Error by month

In [7]:
m["month"] = m["time"].dt.month
by_month = m.groupby("month")["err_mw"].agg(rmse=lambda e: np.sqrt((e ** 2).mean()), bias="mean").round(0)
by_month.T

month,1,2,3,4,5,6,7,8,9,10,11,12
rmse,549.0,624.0,838.0,1373.0,1542.0,1312.0,1476.0,1407.0,1334.0,1299.0,732.0,626.0
bias,47.0,31.0,-2.0,-16.0,-95.0,51.0,-8.0,-20.0,10.0,21.0,-52.0,37.0


Summer months are noisier, which is expected from convective conditions.

## Daily energy

Hours where the forecast is below cut-in are not scheduled, so they are excluded from the
daily energy calculation.

In [8]:
sched = m[m["v_fc"] >= CUT_IN].set_index("time")
daily_mwh = sched["gen_mw"].resample("D").mean() * 24
print(f"scheduled hours: {len(sched):,} of {len(m):,}")
print(f"annual energy 2023: {daily_mwh['2023'].sum() / 1e6:.2f} TWh")
daily_mwh.describe().round(0)

scheduled hours: 17,088 of 17,520
annual energy 2023: 69.21 TWh


count       730.0
mean     194013.0
std       53638.0
min       14758.0
25%      158827.0
50%      216032.0
75%      238958.0
max      240000.0
Name: gen_mw, dtype: float64

## Imbalance exposure

Forecast error is settled at the imbalance price. Use the average premium over day-ahead
observed on the desk (about 20 EUR/MWh) times the typical error.

In [9]:
premium = rng.lognormal(np.log(20), 0.5, len(m))          # EUR/MWh, imbalance premium over day-ahead
exposure_eur = rmse_mw * premium.mean() * 8760
revenue_eur = (m["gen_mw"] * df["price_eur_mwh"].mean()).sum() / 2      # per year
print(f"average premium: {premium.mean():.1f} EUR/MWh")
print(f"annual imbalance exposure: {exposure_eur / 1e6:,.1f} m EUR")
print(f"annual revenue at average price: {revenue_eur / 1e6:,.0f} m EUR")
print(f"exposure / revenue: {exposure_eur / revenue_eur:.1%}")

average premium: 22.9 EUR/MWh
annual imbalance exposure: 231.6 m EUR
annual revenue at average price: 6,911 m EUR
exposure / revenue: 3.4%


## Results

In [10]:
summary = pd.Series({
    "capacity factor": round(cf, 3),
    "forecast RMSE (% of average output)": round(rmse_pct, 3),
    "calibration slope": round(lr.coef_[0], 3),
    "annual energy 2023 (TWh)": round(daily_mwh["2023"].sum() / 1e6, 2),
    "imbalance exposure (m EUR / yr)": round(exposure_eur / 1e6, 1),
    "exposure as % of revenue": round(exposure_eur / revenue_eur, 3),
})
print(summary.to_string())
print()
print("Forecast error is ~11% of output and exposure is about 3% of revenue: imbalance risk is")
print("small relative to the PPA price; no separate risk premium is needed.")

capacity factor                          0.801
forecast RMSE (% of average output)      0.115
calibration slope                        0.920
annual energy 2023 (TWh)                69.210
imbalance exposure (m EUR / yr)        231.600
exposure as % of revenue                 0.034

Forecast error is ~11% of output and exposure is about 3% of revenue: imbalance risk is
small relative to the PPA price; no separate risk premium is needed.
